In [ ]:
# Cell 1: Imports, callback, cleanup, baseline training, and JSON export
from pathlib import Path
import json
import re

import gymnasium as gym
import torch
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import BaseCallback
from stable_baselines3.common.monitor import Monitor

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"Device: {'cuda' if torch.cuda.is_available() else 'cpu'}")

thresholds = [-195, -190, -180, -170, -160]

class MultiThresholdCallback(BaseCallback):
    def __init__(self, thresholds=None, verbose=0):
        super().__init__(verbose)
        self.thresholds = thresholds or [-195, -190, -180, -170, -160]
        self.threshold_steps = {t: None for t in self.thresholds}

    def _on_step(self) -> bool:
        for info in self.locals.get("infos", []):
            if "episode" in info:
                episode_reward = info["episode"]["r"]
                for threshold in self.thresholds:
                    if episode_reward >= threshold and self.threshold_steps[threshold] is None:
                        self.threshold_steps[threshold] = self.num_timesteps
                        print(f"Baseline reached threshold {threshold} at step {self.num_timesteps}")
        return True

log_dir = Path("logs_and_results")
tensorboard_dir = Path("tensorboard_logs")
log_dir.mkdir(parents=True, exist_ok=True)

env = gym.make("MountainCar-v0", render_mode="rgb_array")
env = Monitor(env)

model = PPO(
    policy="MlpPolicy",
    env=env,
    learning_rate=0.0003,
    n_steps=2048,
    batch_size=64,
    n_epochs=10,
    gamma=0.99,
    gae_lambda=0.95,
    clip_range=0.2,
    verbose=1,
    tensorboard_log=str(log_dir / "tensorboard_logs"),
    device="cuda" if torch.cuda.is_available() else "cpu",
)

callback = MultiThresholdCallback(thresholds=thresholds)
print("Start baseline training for 600000 timesteps")
model.learn(
    total_timesteps=600000,
    callback=callback,
    tb_log_name="ppo_baseline_multi_thresh",
)

model_path = log_dir / "baseline_multi_thresh"
model.save(str(model_path))
print(f"Saved model: {model_path}.zip")

baseline_results = {str(t): (s if s is not None else 600000) for t, s in callback.threshold_steps.items()}
json_path = log_dir / "baseline_thresholds.json"
with open(json_path, "w", encoding="utf-8") as f:
    json.dump(baseline_results, f, indent=2)
print(f"Saved threshold metrics: {json_path}")

env.close()

Using device: cuda


In [ ]:
# Cell 2: Evaluate trained baseline model
import imageio.v2 as imageio
from IPython.display import Video
import numpy as np

model_path = "logs_and_results/baseline_multi_thresh.zip"
video_path = "logs_and_results/baseline_multi_thresh_demo.mp4"

loaded_model = PPO.load(model_path)
video_env = gym.make("MountainCar-v0", render_mode="rgb_array")

best_frames = []
best_reward = -np.inf

print("Evaluate model and collect best episode")
for i in range(5):
    current_frames = []
    total_reward = 0.0
    obs, info = video_env.reset()
    terminated = False
    truncated = False

    while not (terminated or truncated):
        current_frames.append(video_env.render())
        action, _ = loaded_model.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = video_env.step(action)
        total_reward += reward

    print(f"Attempt {i + 1}: reward = {total_reward}")
    if total_reward > best_reward:
        best_reward = total_reward
        best_frames = current_frames

print(f"Best reward: {best_reward}")
imageio.mimsave(video_path, best_frames, fps=30)
video_env.close()
Video(video_path, embed=True)